# HPC-Vision — CUDA benchmark runner

Measures the naive and shared-memory-tiled matrix multiplication kernels
from `cuda/` across a size sweep, and writes `results/cuda_naive_results.csv`
and `results/cuda_tiled_results.csv`.

Runs in Google Colab because Apple Silicon has no CUDA runtime.
**Runtime → Change runtime type → T4 GPU** before running.

Method notes, because they affect the numbers:

- Timing uses `cudaEvent` around the kernel only. Host/device transfer
  and allocation are excluded, so these are kernel times, not end-to-end times.
- One warm-up launch is discarded — the first launch pays context-creation
  and JIT cost that is not part of steady-state performance.
- Each size is the mean of `REPS` launches rather than a single shot.
- Correctness is asserted per run: both inputs are all-ones, so every
  output element must equal `n`.


In [ ]:
!nvidia-smi
!nvcc --version


## Kernels

Same algorithms as the committed `.cu` files, with `N` parameterised and
event timing added so one binary can run the whole sweep.


In [ ]:
%%writefile bench_cuda.cu
// Timing harness for the two kernels in cuda/. Same algorithms as
// matrix_cuda_naive.cu and matrix_cuda_tiled.cu, with N taken from argv and
// cudaEvent timing added so a size sweep can be measured in one run.
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void matrixMulNaive(const float *A, const float *B, float *C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; k++) sum += A[row * n + k] * B[k * n + col];
        C[row * n + col] = sum;
    }
}

__global__ void matrixMulTiled(const float *A, const float *B, float *C, int n) {
    __shared__ float tileA[TILE][TILE];
    __shared__ float tileB[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t = 0; t < n / TILE; t++) {
        tileA[threadIdx.y][threadIdx.x] = A[row * n + t * TILE + threadIdx.x];
        tileB[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * n + col];
        __syncthreads();
        for (int k = 0; k < TILE; k++) sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        __syncthreads();
    }
    if (row < n && col < n) C[row * n + col] = sum;
}

int main(int argc, char **argv) {
    int n      = (argc > 1) ? atoi(argv[1]) : 512;
    int tiled  = (argc > 2) ? atoi(argv[2]) : 0;
    int reps   = (argc > 3) ? atoi(argv[3]) : 10;
    size_t bytes = (size_t)n * n * sizeof(float);

    float *hA = (float *)malloc(bytes), *hB = (float *)malloc(bytes), *hC = (float *)malloc(bytes);
    for (int i = 0; i < n * n; i++) { hA[i] = 1.0f; hB[i] = 1.0f; }

    float *dA, *dB, *dC;
    cudaMalloc(&dA, bytes); cudaMalloc(&dB, bytes); cudaMalloc(&dC, bytes);
    cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice);

    dim3 threads(TILE, TILE);
    dim3 blocks((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);

    // Warm-up: the first launch pays context and JIT costs that are not part
    // of steady-state kernel time.
    if (tiled) matrixMulTiled<<<blocks, threads>>>(dA, dB, dC, n);
    else       matrixMulNaive<<<blocks, threads>>>(dA, dB, dC, n);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    for (int r = 0; r < reps; r++) {
        if (tiled) matrixMulTiled<<<blocks, threads>>>(dA, dB, dC, n);
        else       matrixMulNaive<<<blocks, threads>>>(dA, dB, dC, n);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost);
    // Every element is a dot product of two all-ones vectors, so C[0] must be n.
    if (hC[0] != (float)n) fprintf(stderr, "CORRECTNESS FAIL: C[0]=%f expected %d\n", hC[0], n);

    // Seconds per single kernel launch, matching the units in results/*.csv
    printf("%d,%.9f\n", n, (ms / 1000.0) / reps);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(hA); free(hB); free(hC);
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 bench_cuda.cu -o bench_cuda
print('compiled')


## Sweep

Sizes are multiples of the 16-wide tile, so the tiled kernel needs no
boundary handling and the two are compared on equal terms.


In [ ]:
import subprocess, pathlib

SIZES = [256, 512, 1024, 1536, 2048, 3072]
REPS  = 10

pathlib.Path('results').mkdir(exist_ok=True)

for tiled, name in ((0, 'cuda_naive'), (1, 'cuda_tiled')):
    rows = ['size,time']
    for n in SIZES:
        out = subprocess.run(
            ['./bench_cuda', str(n), str(tiled), str(REPS)],
            capture_output=True, text=True, check=True,
        )
        if out.stderr.strip():
            raise RuntimeError(f'{name} n={n}: {out.stderr.strip()}')
        rows.append(out.stdout.strip())
        print(f'{name:11} n={n:5}  {out.stdout.strip().split(chr(44))[1]} s')
    pathlib.Path(f'results/{name}_results.csv').write_text(chr(10).join(rows) + chr(10))
    print(f'wrote results/{name}_results.csv')


## Download

Commit both CSVs to `results/` in the repository. Full timer precision —
do not round: the precision is what distinguishes a measurement from an
estimate.


In [ ]:
from google.colab import files
files.download('results/cuda_naive_results.csv')
files.download('results/cuda_tiled_results.csv')
